<a href="https://colab.research.google.com/github/1nf1n1tee/transformer_custom/blob/main/Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install transformers torch

# Tokenization

In [30]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [25]:
# Sample sentence
sample_sentence = "A flying bird sits on the tree"

# Tokenize the sentence
tokens = tokenizer(sample_sentence)

# The input IDs are the numerical representations of the tokens
input_ids = tokens["input_ids"]

print("Original sentence:", sample_sentence)
print("Tokens:", tokens)
print("Input IDs:", input_ids)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Original sentence: A flying bird sits on the tree
Tokens: {'input_ids': [101, 1037, 3909, 4743, 7719, 2006, 1996, 3392, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1]}
Input IDs: [101, 1037, 3909, 4743, 7719, 2006, 1996, 3392, 102]


In [26]:
def get_token_ids(sentence,tokenizer):
  """
  Tokenizes a sentence using the loaded tokenizer and returns the input IDs as a list.

  Args:
    sentence: The input string sentence.

  Returns:
    A list of token IDs.
  """
  tokens = tokenizer(sentence)
  return tokens["input_ids"]

# Example usage:
sample_sentence = "A flying bird sits on the tree"
token_ids_list = get_token_ids(sample_sentence,tokenizer)
print(f"Original sentence: \"{sample_sentence}\"")
print(f"Token IDs list: {token_ids_list}")

Original sentence: "A flying bird sits on the tree"
Token IDs list: [101, 1037, 3909, 4743, 7719, 2006, 1996, 3392, 102]


# Embedding

In [28]:
import torch
import torch.nn as nn

# vocab_size = len(word_to_index) # This is based on our small vocabulary
embedding_dim = 128

# Use a larger vocabulary size suitable for the transformers tokenizer (e.g., bert-base-uncased has ~30k tokens)
# We can use a common size like 50000 to be safe.
embedding_layer = nn.Embedding(50000, embedding_dim)

print(embedding_layer)

Embedding(50000, 128)


## Implement positional encoding

### Subtask:
Implement positional encoding.


In [20]:
import torch

def positional_encoding(seq_len, d_model):
  """
  Calculates positional encodings for a sequence.

  Args:
    seq_len: The length of the input sequence.
    d_model: The embedding dimension.

  Returns:
    A PyTorch tensor of shape (seq_len, d_model) containing positional encodings.
  """
  pe = torch.zeros(seq_len, d_model)
  position = torch.arange(0, seq_len, dtype=torch.float).unsqueeze(1)
  div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-torch.log(torch.tensor(10000.0)) / d_model))
  pe[:, 0::2] = torch.sin(position * div_term)
  pe[:, 1::2] = torch.cos(position * div_term)
  return pe

# Example usage:
seq_len = 10
d_model = 128
pe = positional_encoding(seq_len, d_model)
print(pe)
print(pe.shape)

tensor([[ 0.0000e+00,  1.0000e+00,  0.0000e+00,  ...,  1.0000e+00,
          0.0000e+00,  1.0000e+00],
        [ 8.4147e-01,  5.4030e-01,  7.6172e-01,  ...,  1.0000e+00,
          1.1548e-04,  1.0000e+00],
        [ 9.0930e-01, -4.1615e-01,  9.8705e-01,  ...,  1.0000e+00,
          2.3096e-04,  1.0000e+00],
        ...,
        [ 6.5699e-01,  7.5390e-01, -2.1963e-01,  ...,  1.0000e+00,
          8.0835e-04,  1.0000e+00],
        [ 9.8936e-01, -1.4550e-01,  6.0082e-01,  ...,  1.0000e+00,
          9.2383e-04,  1.0000e+00],
        [ 4.1212e-01, -9.1113e-01,  9.9818e-01,  ...,  1.0000e+00,
          1.0393e-03,  1.0000e+00]])
torch.Size([10, 128])


## Combine embedding and positional encoding

### Subtask:
Create a function that takes a sequence of word indices, retrieves their embeddings from the embedding layer, and adds the corresponding positional encodings.


**Reasoning**:
Define the function `combine_embedding_and_positional_encoding` that takes word indices and the embedding layer, converts indices to a tensor, gets embeddings, calculates positional encodings, adds them, and returns the result.



In [31]:
import torch

def combine_embedding_and_positional_encoding(word_indices, embedding_layer):
  """
  Combines word embeddings with positional encodings.

  Args:
    word_indices: A list of word indices.
    embedding_layer: The embedding layer object.

  Returns:
    A PyTorch tensor with combined embeddings and positional encodings.
  """
  word_indices_tensor = torch.tensor(word_indices)
  word_embeddings = embedding_layer(word_indices_tensor)
  seq_len = word_embeddings.shape[0]
  d_model = word_embeddings.shape[1]
  pos_encoding = positional_encoding(seq_len, d_model)
  combined_embedding = word_embeddings + pos_encoding
  return combined_embedding


## Test the process

### Subtask:
Use a sample sentence from your text, convert it to indices, and pass it through the combined embedding and positional encoding steps. Display the resulting vectors.


**Reasoning**:
Choose a sample sentence, convert it to indices, and pass it through the combined embedding and positional encoding function. Then print the resulting tensor and its shape.



In [29]:
# 1. Choose a sample sentence from the original text.
sample_sentence = "A flying bird sits on the tree"
sample_words = sample_sentence.split()

# 2. Convert the words in the sample sentence to their corresponding indices using the word_to_index dictionary.
# Handle words not in the vocabulary by skipping them or using a default index if available
# sample_indices = [word_to_index[word] for word in sample_words if word in word_to_index]
sample_indices = get_token_ids(sample_sentence,tokenizer)

# Print the generated indices to inspect them
print("Generated sample indices:", sample_indices)
# Expected shape should be (number_of_words_in_sentence, embedding_dimension)
# In this case, 7 words and embedding_dimension is 128, so (7, 128)


# 3. Call the combine_embedding_and_positional_encoding function with the list of word indices and the embedding_layer.
combined_vectors = combine_embedding_and_positional_encoding(sample_indices, embedding_layer)

# 4. Print the resulting tensor containing the combined embeddings and positional encodings.
print("Combined Embeddings and Positional Encodings for the sample sentence:")
print(combined_vectors)

# 5. Print the shape of the resulting tensor to verify its dimensions.
print("\nShape of the combined vectors:", combined_vectors.shape)

Generated sample indices: [101, 1037, 3909, 4743, 7719, 2006, 1996, 3392, 102]
Combined Embeddings and Positional Encodings for the sample sentence:
tensor([[ 0.9636,  0.4047,  0.5683,  ...,  0.7781,  1.1225,  1.5861],
        [ 0.7367, -0.4313,  0.4368,  ...,  2.2738,  0.8325,  0.2748],
        [-0.3158, -1.4954,  0.0342,  ...,  2.3084, -0.7513, -0.5514],
        ...,
        [-1.0538,  1.0731, -1.1952,  ...,  2.4172,  0.5358,  0.5115],
        [ 1.0695, -0.1378, -0.9452,  ...,  0.4701, -0.1662, -0.4710],
        [ 2.3626,  0.2240,  0.1707,  ...,  0.1408,  0.7762,  1.7990]],
       grad_fn=<AddBackward0>)

Shape of the combined vectors: torch.Size([9, 128])
